# Phase 2 — Preprocessing and Logistic Regression

Predict **normal (0)** versus **attack (1)** using the headered UNSW-NB15 training file. We reserve the official testing file for a later final evaluation. Our 20% held-out subset is called validation in saved outputs; it plays the test-set role for this learning exercise. Do not use its scores to tune the model repeatedly.

This notebook uses the existing reusable helpers. Run all cells from the repository root or `notebooks/`.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Start Jupyter from the project root or notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from src.data_loader import load_dataset
from src.preprocessing import (separate_features_target, identify_feature_types,
    clean_observations, split_dataset, build_preprocessor)
from src.evaluation import calculate_metrics, print_classification_report, plot_confusion_matrix
from src.utils import RANDOM_STATE

## 1. Load and inspect the target

`label` is the binary target. `attack_cat` describes the outcome and would leak the answer into the model. `id` is a record identifier. Drop these two inputs for documented reasons; retain all 42 traffic features. Integer counts and binary flags remain numerical for this baseline.

In [ ]:
raw = load_dataset("training")
print("Shape:", raw.shape)
print(raw["label"].value_counts().sort_index())
print(raw["label"].value_counts(normalize=True).sort_index())
X_raw, y_raw = separate_features_target(raw)
numeric, categorical = identify_feature_types(X_raw)
print("Numerical columns:", numeric)
print("Categorical columns:", categorical)
print("Missing values by column:\n", raw.isna().sum())
print("Full-row duplicates:", raw.duplicated().sum())
print("Repeated feature rows:", X_raw.duplicated().sum())
print("Numeric infinities:", np.isinf(X_raw[numeric]).sum().sum())
for column in categorical:
    print(column, X_raw[column].value_counts().to_dict())

## 2. Clean observations and separate X and y

Unique IDs hide repeated traffic inputs. If repeated inputs land in training and validation, the evaluation can become optimistic. Exclude all rows in groups with contradictory labels, then retain one row per remaining feature combination. This is a transparent baseline choice, not proof that repeated traffic is erroneous. It changes class balance and removes frequency information. Later experiments could preserve frequencies using grouped splits.

Numeric infinities become missing values. No observed range is declared invalid without evidence. Zero remains valid. The service category `-` is retained as no identified service; it is not blindly replaced with a missing value. Actual missing values will be imputed **after splitting**.

In [ ]:
X, y, audit = clean_observations(raw)
print(audit)
print("Clean class counts:", y.value_counts().sort_index().to_dict())
print("Clean class proportions:", y.value_counts(normalize=True).sort_index().to_dict())
assert not X.duplicated().any()
assert not {"label", "attack_cat", "id"}.intersection(X.columns)

## 3. Split before learning any transformation

Use 80% for training and 20% for held-out validation, with `RANDOM_STATE = 42`. Stratification keeps both class proportions similar. Deterministic duplicate and invalid-value checks above learn no imputation statistics or scaling parameters. Median values, means, scales, and category vocabulary must learn only from training data. Learning these from the whole dataset would let held-out information influence training: **data leakage**.

In [ ]:
X_train, X_test, y_train, y_test = split_dataset(X, y)
numeric, categorical = identify_feature_types(X_train)
print("Train:", X_train.shape, "Validation:", X_test.shape)
print("Train labels:", y_train.value_counts().sort_index().to_dict())
print("Validation labels:", y_test.value_counts().sort_index().to_dict())
assert set(X_train.index).isdisjoint(X_test.index)

## 4. Build preprocessing and train the baseline

Numerical inputs: fill missing values with the **training median**, then standardize using training means and scales. Categorical inputs: fill with the most frequent training category, then one-hot encode. `handle_unknown="ignore"` safely gives unseen categories all-zero indicators for that feature.

The `ColumnTransformer` handles the two feature types. The outer `Pipeline` joins preparation and Logistic Regression so fit and predict use the same rules. Logistic Regression estimates an attack probability with a linear decision boundary in the transformed features. It is a useful, understandable baseline. Scaling helps optimization. Use simple defaults, `max_iter=2000`, and no tuning or class weighting.

In [ ]:
pipeline = Pipeline([
    ("preprocessing", build_preprocessor(numeric, categorical)),
    ("classifier", LogisticRegression(random_state=RANDOM_STATE, max_iter=2000)),
])
pipeline.fit(X_train, y_train)
print("Optimizer iterations:", pipeline.named_steps["classifier"].n_iter_)

## 5. Predict and evaluate

`predict` uses the default probability threshold of 0.5. Precision, recall, and F1 below treat **attack** as positive. Accuracy alone may reward a majority-class guess; it also hides the different costs of false alarms and missed attacks. Read the class-specific report and confusion matrix as well. ROC-AUC measures probability ranking over thresholds, not operational detection reliability.

In [ ]:
predictions = pipeline.predict(X_test)
probabilities = pipeline.predict_proba(X_test)[:, 1]
metrics = calculate_metrics(y_test, predictions, probabilities)
display(pd.DataFrame([metrics]))
report = print_classification_report(y_test, predictions)
figure = plot_confusion_matrix(y_test, predictions)
plt.show()

## 6. Save small outputs and the complete pipeline

Save the preprocessing **and** classifier together. Generated artifacts are ignored by Git; source and findings remain reviewable. Only load joblib files you trust.

In [ ]:
for directory in ["results/metrics", "results/figures", "models"]:
    (ROOT / directory).mkdir(parents=True, exist_ok=True)
pd.DataFrame([metrics]).to_csv(ROOT / "results/metrics/logistic_regression_metrics.csv", index=False)
(ROOT / "results/metrics/logistic_regression_classification_report.txt").write_text(report)
figure.savefig(ROOT / "results/figures/logistic_regression_confusion_matrix.png", dpi=150)
joblib.dump(pipeline, ROOT / "models/logistic_regression_pipeline.joblib")
plt.close(figure)

## 7. Interpret this run

The initial run gave 90.07% accuracy, 84.45% attack precision, 97.60% attack recall, 90.55% F1, and 0.96824 ROC-AUC. The confusion matrix contained 8,567 true normals, 1,766 false alarms, 236 missed attacks, and 9,594 detected attacks. Compare your live outputs above with `docs/findings.md`.

Raw attacks were 68.06% of observations. After excluding ambiguous groups and retaining unique inputs, attacks were 48.76%. Cleaning changed the population; these scores cannot be represented as accuracy on the original traffic stream. Exact feature overlap is removed and the known label leakage is excluded. Related traffic patterns may still occur in both subsets. These strong scores do not prove generalization to future networks, different capture conditions, TLS 1.3, or post-quantum traffic. The official test set has not been evaluated.

## Learning notes

- **Features:** input measurements such as packet counts or protocol.
- **Target/label:** the answer we want to predict; here normal or attack.
- **Training set:** observations used to learn model parameters and preprocessing statistics.
- **Test set:** held-out observations used to assess predictions. Here we use an internal validation subset and reserve the official test file.
- **Categorical feature:** a named group, such as protocol.
- **Numerical feature:** a number measuring a quantity, count, or indicator.
- **Encoding:** turning categories into numerical inputs, here one indicator per category.
- **Feature scaling:** bringing numerical features onto comparable scales using training statistics.
- **Data leakage:** information from labels or held-out data improperly influences training.
- **Stratified split:** a split that keeps class proportions approximately the same.
- **Logistic Regression:** a linear classifier that estimates class probabilities despite its name.
- **Precision:** of the rows flagged as attacks, how many really are attacks?
- **Recall:** of all actual attacks, how many did we detect?
- **F1-score:** the harmonic mean of precision and recall; it balances the two.
- **Confusion matrix:** counts of correct and incorrect predictions by true class and predicted class.
- **False positive:** normal traffic flagged as attack; a false alarm that costs analyst attention.
- **False negative:** an attack classified as normal; a missed intrusion.

Before Phase 3, explain the fit/predict distinction, the split, the two error types, and the population change from cleaning. Phase 3 could compare a single Random Forest baseline on the same split and metrics, checking overfitting and errors. This notebook stops at Phase 2.

Clear notebook outputs before committing. Run `python -m src.train_baseline` for the same experiment plus a JSON audit, dataset checksum, versions, and convergence status.